# Where Python Actually Starts
A Python file looks calm on the screen, but the moment execution begins the interpreter starts building frames, loading constants, resolving names, and moving through bytecode one instruction at a time. This notebook begins from that living runtime picture so that later topics feel connected instead of scattered.

When people struggle with **Where Python Actually Starts**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- See the difference between source code, bytecode, and runtime execution
- Understand what a frame is in practical terms
- Separate names from objects early
- Build a mental picture that will support everything else


At runtime, Python keeps namespaces that map names to objects. A frame carries local names, links to global names, and information about where execution currently is. The important human insight is this: values do not float around alone. They are reached through names, containers, attributes, and call frames.


In [1]:
x = 42
text = "python"
values = [x, text]
print("id(x):", id(x))
print("id(text):", id(text))
print("id(values):", id(values))
print(values)


id(x): 140715746449480
id(text): 2688898746352
id(values): 2688950250688
[42, 'python']


Bytecode is not something you must memorize, but it is the best way to see that Python lowers your code into smaller runtime instructions. Seeing it once makes the interpreter feel less mystical and more concrete.


In [2]:
import dis

def total(n):
    s = 0
    for i in range(n):
        s += i
    return s

dis.dis(total)


  3           0 RESUME                   0

  4           2 LOAD_CONST               1 (0)
              4 STORE_FAST               1 (s)

  5           6 LOAD_GLOBAL              1 (NULL + range)
             18 LOAD_FAST                0 (n)
             20 PRECALL                  1
             24 CALL                     1
             34 GET_ITER
        >>   36 FOR_ITER                 7 (to 52)
             38 STORE_FAST               2 (i)

  6          40 LOAD_FAST                1 (s)
             42 LOAD_FAST                2 (i)
             44 BINARY_OP               13 (+=)
             48 STORE_FAST               1 (s)
             50 JUMP_BACKWARD            8 (to 36)

  7     >>   52 LOAD_FAST                1 (s)
             54 RETURN_VALUE


The file or notebook cell you write is only the human-friendly layer. Python still has to parse it and turn it into executable instructions.

Whenever you call a function, Python creates a new execution frame. That frame holds local variables and remembers where control should return afterward.

The name `x` is not the integer itself. It is a reference in some namespace that points to an object.

Many “advanced” Python facts stop feeling advanced once you keep the frame and object model in your head.


Each call to `describe` gets its own local variables, even though the function body is the same.


In [3]:
def describe(value):
    local_message = f"inside with {value!r}"
    print(local_message)

describe(10)
describe("python")


inside with 10
inside with 'python'


This exception is intentional. The traceback is showing the chain of frames that led to the problem.


In [4]:
def a():
    b()

def b():
    c()

def c():
    raise ValueError("trace me")

try:
    a()
except ValueError as exc:
    import traceback
    traceback.print_exc()


Traceback (most recent call last):
  File "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_15832\2221454973.py", line 11, in <module>
    a()
  File "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_15832\2221454973.py", line 2, in a
    b()
  File "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_15832\2221454973.py", line 5, in b
    c()
  File "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_15832\2221454973.py", line 8, in c
    raise ValueError("trace me")
ValueError: trace me


A function carries a code object that stores compiled information about the function body.


In [5]:
def sample(a, b):
    return a + b

print(sample.__code__)
print(sample.__code__.co_varnames)


<code object sample at 0x0000027211EDC5E0, file "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_15832\2060303189.py", line 1>
('a', 'b')


This is not only a classroom topic. **Where Python Actually Starts** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Thinking a variable is the same thing as the object it names
- Treating bytecode as irrelevant trivia instead of a useful mental tool
- Ignoring frames when reading tracebacks


- What happens between Python source code and execution?
- What is a frame?
- What is the difference between a name and an object?


- Python executes compiled instructions, not raw text.
- Frames are the working spaces of execution.
- Names point to objects through namespaces.


If this notebook did its job, **Where Python Actually Starts** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.


A better way to understand Where Python Actually Starts is to keep moving back and forth between three views of the same code.

The first view is the human view. This is what the code is trying to say. The second view is the object view. In that view, we ask which Python objects exist, which names refer to them, and which objects are being created, reused, mutated, or discarded. The third view is the interpreter view. In that view, we ask what protocol is being used, what bytecode is being executed, and which runtime rules are controlling the result.

Many learners stop after the first view. That is enough to copy code, but it is usually not enough to explain surprising behavior. Deep understanding starts when these three views become one mental movement instead of three unrelated topics.


There is also a fourth habit that helps a lot: failure-based thinking.

For Where Python Actually Starts, ask yourself what normally goes wrong when the idea is only half-understood. Does someone confuse identity with equality? Do they expect copying where only rebinding happened? Do they expect a fresh iterator when they really have an exhausted one? Do they trust a default value that is secretly shared? Do they think a class attribute is per-instance state? Once you can predict the common failure modes before they happen, your understanding becomes much more stable.


In [6]:
import dis
import inspect

def probe_runtime(value):
    local_copy = value
    frame = inspect.currentframe()
    print('locals now:', frame.f_locals)
    return local_copy

print(probe_runtime('demo'))
print('\nbytecode for probe_runtime:')
dis.dis(probe_runtime)


locals now: {'value': 'demo', 'local_copy': 'demo', 'frame': <frame at 0x0000027211CF7E80, file 'C:\\Users\\VIHAAN\\AppData\\Local\\Temp\\ipykernel_15832\\939914315.py', line 7, code probe_runtime>}
demo

bytecode for probe_runtime:
  4           0 RESUME                   0

  5           2 LOAD_FAST                0 (value)
              4 STORE_FAST               1 (local_copy)

  6           6 LOAD_GLOBAL              0 (inspect)
             18 LOAD_METHOD              1 (currentframe)
             40 PRECALL                  0
             44 CALL                     0
             54 STORE_FAST               2 (frame)

  7          56 LOAD_GLOBAL              5 (NULL + print)
             68 LOAD_CONST               1 ('locals now:')
             70 LOAD_FAST                2 (frame)
             72 LOAD_ATTR                3 (f_locals)
             82 PRECALL                  2
             86 CALL                     2
             96 POP_TOP

  8          98 LOAD_FAST        

A deeper layer here is to stop thinking about execution as a smooth blur. Execution really advances through concrete steps. Python creates a frame, resolves names, loads constants, calls functions, stores results, and returns values. That is why stack traces, code objects, and disassembly are so valuable. They give you snapshots of those steps instead of leaving everything hidden behind the surface syntax.

This is also the right place to begin separating language guarantees from CPython implementation details. The language explains the meaning of the code. CPython explains many of the practical low-level behaviors you observe while running that code.


In [7]:
import inspect
import dis

def execution_story(a, b):
    total = a + b
    frame = inspect.currentframe()
    print('code object name:', frame.f_code.co_name)
    print('line number inside function:', frame.f_lineno)
    print('locals inside frame:', frame.f_locals)
    return total

print(execution_story(7, 8))
print('\nbytecode:')
dis.dis(execution_story)


code object name: execution_story
line number inside function: 8
locals inside frame: {'a': 7, 'b': 8, 'total': 15, 'frame': <frame at 0x0000027211EDE330, file 'C:\\Users\\VIHAAN\\AppData\\Local\\Temp\\ipykernel_15832\\2259766311.py', line 9, code execution_story>}
15

bytecode:
  4           0 RESUME                   0

  5           2 LOAD_FAST                0 (a)
              4 LOAD_FAST                1 (b)
              6 BINARY_OP                0 (+)
             10 STORE_FAST               2 (total)

  6          12 LOAD_GLOBAL              0 (inspect)
             24 LOAD_METHOD              1 (currentframe)
             46 PRECALL                  0
             50 CALL                     0
             60 STORE_FAST               3 (frame)

  7          62 LOAD_GLOBAL              5 (NULL + print)
             74 LOAD_CONST               1 ('code object name:')
             76 LOAD_FAST                3 (frame)
             78 LOAD_ATTR                3 (f_code)
        

One useful way to deepen this chapter is to stop treating execution as a single mysterious event and instead see it as a chain of smaller, understandable events. Python reads source, compiles it, stores constants, creates code objects, enters frames, resolves names, and returns values. None of those pieces are random. They are simply easy to ignore until a bug, a traceback, or a performance question forces you to care. Once you have that layered picture, you can read later notebooks with much more confidence because you are no longer depending on surface syntax alone.


## Final Takeaway

The deepest way to revise **Where Python Actually Starts** is to stop treating it as a collection of syntax facts and instead treat it as a runtime story. Ask what objects are involved, what names or attributes point to those objects, what frame is active, and what protocol or bytecode path Python is following. If you can explain this topic at those four levels, then you understand much more than the visible syntax.

In interview terms, the strongest answer is usually not the shortest definition. It is a calm explanation of what Python is doing under the surface and why the behavior follows from that model. For revision, come back to this notebook and ask yourself: could I explain this entire chapter with one small code example, one memory-level explanation, and one interpreter-level explanation? If yes, the topic is becoming yours.
